# Ball-Free Possession Baseline Analysis

This notebook demonstrates the ball-free possession baseline on the first 5 minutes (7500 frames) of Metrica Sample Game 1.

It covers:
1. Loading canonical tracking and event data
2. Generating possession predictions
3. Inferring event candidates (passes, turnovers, recoveries)
4. Evaluating predictions against ground-truth Metrica events
5. Visualising possession timelines and producing figures

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from src.data.metrica_parser import load_metrica_match
from src.data.metrica_event_parser import load_metrica_events
from src.possession.possession_baseline import (
    predict_possession,
    infer_events,
    evaluate_predictions,
    possession_summary
)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Ensure figures directory exists
os.makedirs('../../results/figures', exist_ok=True)

## 1. Load Data

We load the Metrica tracking data and reference event data, restricting our analysis to the first 7,500 frames (5 minutes at 25 FPS) for rapid baseline evaluation.

In [ ]:
# File paths
base_dir = Path('../../data/raw/metrica/data/Sample_Game_1')
home_path = base_dir / 'Sample_Game_1_RawTrackingData_Home_Team.csv'
away_path = base_dir / 'Sample_Game_1_RawTrackingData_Away_Team.csv'
events_path = base_dir / 'Sample_Game_1_RawEventsData.csv'

# Configuration
MATCH_ID = 'sample_game_1'
N_FRAMES_SUBSET = 7500  # 5 minutes at 25 fps
FPS = 25.0

# Load tracking
print("Loading tracking data...")
tracking_df, _ = load_metrica_match(home_path, away_path, MATCH_ID)
tracking_subset = tracking_df[tracking_df['frame'] <= N_FRAMES_SUBSET].copy()
print(f"Loaded {len(tracking_subset)} tracking rows for first {N_FRAMES_SUBSET} frames.")

# Load events
print("Loading reference events...")
reference_events = load_metrica_events(events_path, MATCH_ID)
ref_subset = reference_events[(reference_events['start_frame'] <= N_FRAMES_SUBSET) & (reference_events['period'] == 1)].copy()
print(f"Loaded {len(ref_subset)} reference events in time window.")

## 2. Produce Possession Predictions

We run the heuristic possession baseline on the tracking subset. Notice that we do not pass the ball coordinates or event labels to the prediction function.

In [ ]:
print("Predicting possession (this constructs the spatial graphs internally)...")
possession_df = predict_possession(
    tracking_subset,
    switch_margin=0.15,
    persistence_window=5,
    min_score_threshold=0.10,
    dt=1.0/FPS
)

summary = possession_summary(possession_df)
display(pd.DataFrame([summary]))

## 3. Infer Events and Evaluate

We convert possession transitions into candidate events and evaluate them against the Metrica ground truth.

In [ ]:
print("Inferring candidate events...")
predicted_events = infer_events(possession_df)
print(f"Generated {len(predicted_events)} candidate events.")

print("\nEvaluating against reference events...")
eval_results = evaluate_predictions(predicted_events, ref_subset, fps=FPS, tolerances_sec=[0.5, 1.0, 2.0])
display(eval_results)

## 4. Visualisations

### 4.1 Possession Timeline

We plot a short window to show how the heuristic tracks possession continuity.

In [ ]:
# Select a 30-second window (750 frames)
start_f, end_f = 2500, 3250
window_df = possession_df[(possession_df['frame'] >= start_f) & (possession_df['frame'] <= end_f)].copy()

plt.figure(figsize=(15, 3))

# Map teams to y-values for plotting
y_vals = []
colors = []
for t in window_df['team']:
    if t == 'home':
        y_vals.append(1)
        colors.append('blue')
    elif t == 'away':
        y_vals.append(-1)
        colors.append('red')
    else:
        y_vals.append(0)
        colors.append('grey')

plt.scatter(window_df['timestamp'], y_vals, c=colors, marker='|', s=500, alpha=0.5)
plt.yticks([-1, 0, 1], ['Away', 'None', 'Home'])
plt.xlabel('Time (s)')
plt.title('Possession Timeline (Frames 2500-3250)')
plt.ylim(-1.5, 1.5)
plt.tight_layout()
plt.savefig('../../results/figures/possession_timeline_sample.png', dpi=300)
plt.show()

### 4.2 Precision-Recall Curve (by Tolerance)

Shows how evaluation metrics improve as we loosen the temporal matching tolerance.

In [ ]:
metrics_melted = eval_results.melt(
    id_vars=['event_type', 'tolerance_sec'],
    value_vars=['precision', 'recall', 'f1'],
    var_name='metric',
    value_name='score'
)

plt.figure(figsize=(10, 6))
sns.lineplot(
    data=metrics_melted,
    x='tolerance_sec',
    y='score',
    hue='event_type',
    style='metric',
    markers=True
)
plt.title('Baseline Performance vs. Temporal Tolerance')
plt.xlabel('Tolerance (seconds)')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../../results/figures/possession_baseline_pr_curve.png', dpi=300)
plt.show()

## 5. Qualitative Analysis (Successes and Failures)

We identify examples of correct predictions, false positives, and false negatives to understand where the heuristic fails.

In [ ]:
def print_match_examples(pred_type, ref_type, tol=1.0):
    pred_subset = predicted_events[predicted_events['event_type'] == pred_type]
    ref_subset = ref_subset[ref_subset['event_type'] == ref_type]
    
    tol_frames = int(tol * FPS)
    
    print(f"\n--- Analysis: {pred_type} vs {ref_type} (Tolerance ±{tol}s) ---")
    
    # Match logic (simplified greedy for demo)
    matches = []
    unmatched_preds = list(pred_subset['frame'].values)
    unmatched_refs = list(ref_subset['start_frame'].values)
    
    for p_frame in pred_subset['frame'].values:
        for r_frame in unmatched_refs:
            if abs(p_frame - r_frame) <= tol_frames:
                matches.append((p_frame, r_frame))
                unmatched_preds.remove(p_frame)
                unmatched_refs.remove(r_frame)
                break
                
    if matches:
        print(f"\nLikely Correct Prediction (True Positive):")
        p, r = matches[0]
        print(f"  Predicted at frame {p}, Reference at frame {r} (diff: {(p-r)/FPS:.2f}s)")
        
    if unmatched_preds:
        print(f"\nLikely False Positive (Predicted, but no reference event nearby):")
        p = unmatched_preds[0]
        print(f"  Predicted at frame {p}")
        
    if unmatched_refs:
        print(f"\nLikely False Negative (Reference event missed by heuristic):")
        r = unmatched_refs[0]
        print(f"  Reference at frame {r}")
        
print_match_examples('pass_candidate', 'PASS')
print_match_examples('turnover_candidate', 'BALL LOST')

### Conclusion

This baseline demonstrates that possession can be partially inferred from spatial geometry alone, but it falls short of ground-truth accuracy. It struggles particularly with set pieces (dense clusters), long passes (flight time where no player is nearby), and goalkeepers (often isolated, scoring low on proximity).

Future upgrades (GNNs, CRFs) will use this baseline as a benchmark.